# Multimodal Vectorless RAG with LangGraph + NullVector

This cookbook uses the current NullVector acquisition, projection, and attachment-only multimodal contracts.
It validates the current committed tree before retrieval, blocks polluted local trees visibly, and uses the framework text gateway for any optional live answer generation.


### Environment

- Uses `903000608.pdf` when present, otherwise falls back to the committed Phase 01 born-digital fixture.
- Reuses or regenerates acquisition/tree artifacts with content/settings-derived run IDs.
- Uses LangGraph for orchestration and the NullVector noop multimodal gateway for deterministic visual enrichment.
- Uses the NullVector `GatewayService` for optional live text only when `OPENROUTER_API_KEY` is configured; otherwise answer generation falls back deterministically.
- Any `NODE_SUMMARY` artifacts loaded here were created during tree builds where `summarize=True` batches LLM-needed nodes per level through `invoke_many()`.
- If a local real-PDF tree is obviously polluted, the notebook stops that path and prints a structured failure summary instead of fabricating a grounded answer.


In [ ]:
# environment setup
import hashlib
import json
import os
from pathlib import Path
from typing import Any, TypedDict

REPO_ROOT = Path.cwd()
COOKBOOK_ROOT = REPO_ROOT / "cookbook"
COOKBOOK_ARTIFACT_ROOT = COOKBOOK_ROOT / "artifacts"
COOKBOOK_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_PDF = REPO_ROOT / "903000608.pdf"
FIXTURE_PDF = REPO_ROOT / "fixtures" / "pdfs" / "phase01" / "born_digital_with_outline.pdf"
PDF_PATH = LOCAL_PDF if LOCAL_PDF.exists() else FIXTURE_PDF
PDF_SOURCE_MODE = "local_real_pdf" if LOCAL_PDF.exists() else "fixture_pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"No source PDF available at {LOCAL_PDF} or {FIXTURE_PDF}")

print(
    json.dumps(
        {
            "repo_root": str(REPO_ROOT),
            "pdf_path": str(PDF_PATH),
            "pdf_source_mode": PDF_SOURCE_MODE,
        },
        indent=2,
    )
)

In [ ]:
# imports
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, TypeAdapter

from nullvector.cookbook_support import (
    build_tree_quality_failure,
    evaluate_tree_quality,
    format_grounded_fallback_answer,
    load_typed_json,
)
from nullvector.domain import (
    AcquisitionRequest,
    AcquisitionRunManifest,
    AcquisitionSettings,
    CanonicalDocumentLedger,
    NodeCard,
    NodeSummary,
    TreeBuildManifest,
    TreeBuildRequest,
    TreeSettings,
    UnresolvedRegion,
    VisualArtifact,
    VisualEnrichmentRequest,
    VisualRegionReference,
)
from nullvector.ingest import acquire_document
from nullvector.ingest.fingerprint import fingerprint_document
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayError,
    GatewayRequest,
    GatewayService,
    LLMMessage,
    LLMRole,
    NoopProviderAdapter,
    NoopScriptedResponse,
    enrich_visual_region,
)
from nullvector.storage._serialization import settings_digest as acquisition_settings_digest
from nullvector.tree import build_tree

print("LangGraph + NullVector imports succeeded")

In [ ]:
# configuration
class NotebookAnswerResponse(BaseModel):
    answer: str

def load_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text(encoding="utf-8"))

def stable_digest(payload: dict[str, Any]) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode(
            "utf-8"
        )
    ).hexdigest()

def tree_settings_digest(settings: TreeSettings) -> str:
    return stable_digest(settings.model_dump(mode="json"))

NODE_CARD_ADAPTER = TypeAdapter(tuple[NodeCard, ...])
NODE_SUMMARY_ADAPTER = TypeAdapter(tuple[NodeSummary, ...])

def choose_visual_attachment_path(region: VisualRegionReference) -> tuple[str | None, str | None]:
    if region.asset_path:
        return region.asset_path, "asset_path"
    if region.page_render_path:
        return region.page_render_path, "page_render_path"
    return None, None

def block_to_region_reference(
    document_id: str, page_index: int, block: VisualArtifact | UnresolvedRegion
) -> VisualRegionReference:
    region_id = block.visual_id if isinstance(block, VisualArtifact) else block.region_id
    image_ref = block.image_ref if isinstance(block, VisualArtifact) else None
    return VisualRegionReference(
        document_id=document_id,
        page_index=page_index,
        region_id=region_id,
        bbox=block.bbox,
        image_ref=image_ref,
        asset_path=block.asset_path,
        page_render_path=block.page_render_path,
        render_dpi=block.render_dpi,
        coordinate_space=block.coordinate_space,
    )

def collect_enrichable_regions(ledger: CanonicalDocumentLedger) -> list[dict[str, Any]]:
    regions: list[dict[str, Any]] = []
    for page in ledger.pages:
        for block in page.blocks:
            if isinstance(block, VisualArtifact):
                if not block.needs_enrichment:
                    continue
                region = block_to_region_reference(ledger.document_id, page.page_index, block)
                attachment_path, attachment_kind = choose_visual_attachment_path(region)
                regions.append(
                    {
                        "page_index": page.page_index,
                        "region": region,
                        "attachment_path": attachment_path,
                        "attachment_kind": attachment_kind,
                        "kind": "visual_artifact",
                        "reason_code": None,
                        "kind_hint": block.kind_hint,
                    }
                )
            elif isinstance(block, UnresolvedRegion):
                region = block_to_region_reference(ledger.document_id, page.page_index, block)
                attachment_path, attachment_kind = choose_visual_attachment_path(region)
                regions.append(
                    {
                        "page_index": page.page_index,
                        "region": region,
                        "attachment_path": attachment_path,
                        "attachment_kind": attachment_kind,
                        "kind": "unresolved_region",
                        "reason_code": block.reason_code,
                        "kind_hint": None,
                    }
                )
    return regions

def acquire_visual_ready_document(
    pdf_path: Path,
) -> tuple[AcquisitionRunManifest, CanonicalDocumentLedger, list[dict[str, Any]], dict[str, Any]]:
    fingerprint = fingerprint_document(str(pdf_path))
    acquisition_settings = AcquisitionSettings()
    settings_tag = acquisition_settings_digest(acquisition_settings)[:10]
    base_run_id = f"cookbook-acquisition-viz-{fingerprint.sha256[:12]}-{settings_tag}"

    def run_with_id(run_id: str) -> AcquisitionRunManifest:
        return acquire_document(
            AcquisitionRequest(
                source_path=str(pdf_path),
                acquisition_run_id=run_id,
                artifact_root=str(COOKBOOK_ARTIFACT_ROOT / "acquisition_runs"),
                settings=acquisition_settings,
            )
        )

    manifest = run_with_id(base_run_id)
    ledger = CanonicalDocumentLedger.model_validate_json(
        Path(manifest.ledger_path).read_text(encoding="utf-8")
    )
    regions = collect_enrichable_regions(ledger)
    enrichable_count = len(regions)
    asset_backed_count = sum(1 for region in regions if region["attachment_path"] is not None)

    if enrichable_count > 0 and asset_backed_count < enrichable_count:
        manifest = run_with_id(f"{base_run_id}-refresh")
        ledger = CanonicalDocumentLedger.model_validate_json(
            Path(manifest.ledger_path).read_text(encoding="utf-8")
        )
        regions = collect_enrichable_regions(ledger)
        enrichable_count = len(regions)
        asset_backed_count = sum(1 for region in regions if region["attachment_path"] is not None)

    validation = {
        "acquisition_run_id": manifest.acquisition_run_id,
        "enrichable_region_count": enrichable_count,
        "asset_backed_region_count": asset_backed_count,
        "assets_complete": asset_backed_count == enrichable_count if enrichable_count else True,
    }
    return manifest, ledger, regions, validation

def build_tree_manifest(acquisition_manifest: AcquisitionRunManifest) -> TreeBuildManifest:
    tree_settings = TreeSettings()
    tree_tag = tree_settings_digest(tree_settings)[:10]
    tree_run_id = (
        f"cookbook-tree-{acquisition_manifest.source_fingerprint.sha256[:12]}-{tree_tag}-base"
    )
    return build_tree(
        TreeBuildRequest(
            acquisition_manifest_path=str(
                Path(acquisition_manifest.artifact_root) / "manifest.json"
            ),
            tree_run_id=tree_run_id,
            settings=tree_settings,
        )
    )

NOTEBOOK_OPENROUTER_API_KEY = ""
NOTEBOOK_OPENROUTER_API_BASE = ""
OPENROUTER_API_KEY = (
    NOTEBOOK_OPENROUTER_API_KEY.strip() or os.getenv("OPENROUTER_API_KEY", "").strip()
)
OPENROUTER_API_BASE = (
    NOTEBOOK_OPENROUTER_API_BASE.strip()
    or os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1").strip()
)
LLM_MODEL = os.getenv("OPENROUTER_MODEL", "openrouter/google/gemini-3.1-flash-lite-preview")
TEXT_LLM_ENABLED = bool(OPENROUTER_API_KEY)
TEXT_GATEWAY = (
    GatewayService(
        GatewayConfig(
            default_model=LLM_MODEL,
            audit=GatewayAuditConfig(
                persist_root=str(COOKBOOK_ARTIFACT_ROOT / "llm-audit"),
            ),
        )
    )
    if TEXT_LLM_ENABLED
    else None
)

print(
    json.dumps(
        {
            "text_llm_enabled": TEXT_LLM_ENABLED,
            "openrouter_api_key_present": TEXT_LLM_ENABLED,
            "text_provider_path": "GatewayService" if TEXT_GATEWAY is not None else "disabled",
            "text_provider_name": "litellm" if TEXT_GATEWAY is not None else None,
        },
        indent=2,
    )
)

In [ ]:
# execution
fingerprint = fingerprint_document(str(PDF_PATH))
acquisition_manifest, canonical_ledger, visual_regions, visual_validation = (
    acquire_visual_ready_document(PDF_PATH)
)
tree_manifest = build_tree_manifest(acquisition_manifest)

tree_manifest_path = str(Path(tree_manifest.artifact_root) / "manifest.json")
manifest_model = TreeBuildManifest.model_validate_json(
    Path(tree_manifest.artifact_root).joinpath("manifest.json").read_text(encoding="utf-8")
)
node_cards = load_typed_json(manifest_model.node_cards_path, NODE_CARD_ADAPTER)
summaries_by_id = {}
if manifest_model.node_summaries_path:
    for summary in load_typed_json(manifest_model.node_summaries_path, NODE_SUMMARY_ADAPTER):
        summaries_by_id[summary.node_id] = summary
quality_report = evaluate_tree_quality(
    node_cards,
    tree_manifest_path=tree_manifest_path,
    page_count=acquisition_manifest.page_count,
)
cookbook_failure = (
    build_tree_quality_failure(tree_manifest_path=tree_manifest_path, quality_report=quality_report)
    if quality_report.status == "rejected"
    else None
)

print(
    json.dumps(
        {
            "document_id": fingerprint.document_id,
            "tree_run_id": manifest_model.tree_run_id,
            "tree_manifest_path": tree_manifest_path,
            "node_card_count": len(node_cards),
            "summary_count": len(summaries_by_id),
            "visual_validation": visual_validation,
            "quality_gate_status": quality_report.status,
            "quality_issue_codes": [issue.code for issue in quality_report.issues],
        },
        indent=2,
    )
)

In [ ]:
# execution
def region_owner(region_page_index: int, cards: tuple[NodeCard, ...]) -> NodeCard | None:
    candidates = [
        card
        for card in cards
        if card.page_span.start_page <= region_page_index <= card.page_span.end_page
    ]
    if not candidates:
        return None
    return sorted(
        candidates,
        key=lambda card: (
            -card.level,
            (card.page_span.end_page - card.page_span.start_page),
            len(card.path),
            card.node_id,
        ),
    )[0]


node_visual_regions: dict[str, list[dict[str, Any]]] = {}
node_details: dict[str, dict[str, Any]] = {}
l1_sections: list[NodeCard] = []
if cookbook_failure is None:
    for item in visual_regions:
        if item["attachment_path"] is None:
            continue
        owner = region_owner(item["page_index"], node_cards)
        if owner is None:
            continue
        item["region"] = item["region"].model_copy(update={"node_id": owner.node_id})
        node_visual_regions.setdefault(owner.node_id, []).append(item)

    for card in node_cards:
        summary = summaries_by_id.get(card.node_id)
        summary_text = (
            summary.summary
            if summary is not None
            else (
                card.summary
                or f"{card.title} (pages {card.page_span.start_page}-{card.page_span.end_page})"
            )
        )
        node_details[card.node_id] = {
            "node_id": card.node_id,
            "title": card.title,
            "path": list(card.path),
            "level": card.level,
            "page_span": {
                "start_page": card.page_span.start_page,
                "end_page": card.page_span.end_page,
            },
            "summary": summary_text,
            "keywords": list(summary.keywords if summary is not None else card.keywords),
            "visual_regions": node_visual_regions.get(card.node_id, []),
        }
    l1_sections = [card for card in node_cards if card.level == 1]
    print(
        json.dumps(
            {
                "graph_status": "ready",
                "l1_section_count": len(l1_sections),
                "nodes_with_visual_regions": sum(
                    1 for regions in node_visual_regions.values() if regions
                ),
                "asset_backed_visual_region_count": sum(
                    len(regions) for regions in node_visual_regions.values()
                ),
            },
            indent=2,
        )
    )
else:
    print(
        json.dumps(
            {
                "graph_status": "blocked_tree_quality",
                "reason_code": cookbook_failure.reason_code,
                "tree_manifest_path": cookbook_failure.tree_manifest_path,
                "sample_rejected_titles": list(cookbook_failure.sample_rejected_titles),
            },
            indent=2,
        )
    )

In [ ]:
# execution
vision_gateway = GatewayService(
    GatewayConfig(
        default_model="cookbook-multimodal-noop",
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ARTIFACT_ROOT / "multimodal-audit")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "visual_region_enrichment": NoopScriptedResponse(
                output_json={
                    "insight": {
                        "summary": "Attachment-backed visual region reviewed through the NullVector gateway.",
                        "labels": ["asset-backed-region", "attachment-only"],
                        "attributes": {"demo": True},
                        "confidence": 0.5,
                    }
                }
            )
        }
    ),
)

def token_set(text: str) -> set[str]:
    return {
        token
        for token in "".join(ch.lower() if ch.isalnum() else " " for ch in text).split()
        if token
    }

def heuristic_select_sections(query: str) -> tuple[list[str], str]:
    query_tokens = token_set(query)
    scored = []
    for card in l1_sections:
        detail = node_details[card.node_id]
        haystack = f"{detail['title']} {detail['summary']} {' '.join(detail['keywords'])}"
        overlap = len(query_tokens & token_set(haystack))
        scored.append((overlap, card.page_span.start_page, card.node_id))
    selected = [
        node_id
        for overlap, _, node_id in sorted(scored, key=lambda item: (-item[0], item[1], item[2]))
        if overlap > 0
    ][:3]
    if not selected:
        selected = [card.node_id for card in l1_sections[: min(2, len(l1_sections))]]
    return selected, "deterministic token-overlap selection"

def heuristic_select_leaf_nodes(query: str, selected_l1_ids: list[str]) -> tuple[list[str], str]:
    query_tokens = token_set(query)
    descendants = [
        detail
        for detail in node_details.values()
        if detail["node_id"] in selected_l1_ids
        or any(
            detail["path"][:1] == node_details[parent_id]["path"][:1]
            for parent_id in selected_l1_ids
        )
    ]
    scored = []
    for detail in descendants:
        haystack = f"{detail['title']} {detail['summary']} {' '.join(detail['keywords'])}"
        overlap = len(query_tokens & token_set(haystack))
        scored.append(
            (overlap, detail["level"], detail["page_span"]["start_page"], detail["node_id"])
        )
    selected = [
        node_id
        for overlap, _, _, node_id in sorted(
            scored, key=lambda item: (-item[0], -item[1], item[2], item[3])
        )
        if overlap > 0
    ][:5]
    if not selected:
        selected = selected_l1_ids[:]
    return selected, "deterministic node drill-down"

In [ ]:
# execution
class RAGState(TypedDict, total=False):
    query: str
    all_node_details: dict[str, dict[str, Any]]
    l1_sections: list[dict[str, Any]]
    selected_l1_ids: list[str]
    selected_leaf_ids: list[str]
    retrieval_reasoning: str
    drill_down_reasoning: str
    retrieval_trace: list[str]
    visual_attachments_by_node: dict[str, list[dict[str, Any]]]
    context_chunks: list[dict[str, Any]]
    citations: list[dict[str, Any]]
    answer: str
    answer_mode: str
    live_text_result: dict[str, Any]


def load_artifacts_node(state: RAGState) -> dict[str, Any]:
    l1_data = []
    for card in l1_sections:
        detail = node_details[card.node_id]
        l1_data.append(
            {
                "node_id": card.node_id,
                "title": detail["title"],
                "summary": detail["summary"][:240],
                "page_span": detail["page_span"],
                "keywords": detail["keywords"],
            }
        )
    return {
        "all_node_details": node_details,
        "l1_sections": l1_data,
        "retrieval_trace": [
            "Loaded acquisition-backed node index and asset-backed visual regions."
        ],
    }


def select_sections_node(state: RAGState) -> dict[str, Any]:
    selected, reasoning = heuristic_select_sections(state["query"])
    return {
        "selected_l1_ids": selected,
        "retrieval_reasoning": reasoning,
        "retrieval_trace": state["retrieval_trace"] + [f"Selected top-level sections: {selected}"],
    }


def drill_down_node(state: RAGState) -> dict[str, Any]:
    selected, reasoning = heuristic_select_leaf_nodes(state["query"], state["selected_l1_ids"])
    return {
        "selected_leaf_ids": selected,
        "drill_down_reasoning": reasoning,
        "retrieval_trace": state["retrieval_trace"] + [f"Selected leaf/context nodes: {selected}"],
    }


def enrich_visuals_node(state: RAGState) -> dict[str, Any]:
    attachments_by_node: dict[str, list[dict[str, Any]]] = {}
    trace = state["retrieval_trace"][:]
    for node_id in state["selected_leaf_ids"]:
        regions = node_details[node_id].get("visual_regions", [])
        if not regions:
            continue
        for region_item in regions[:1]:
            try:
                attachment = enrich_visual_region(
                    vision_gateway,
                    VisualEnrichmentRequest(
                        request_id=f"langgraph-visual-{region_item['region'].region_id}",
                        region=region_item["region"],
                        prompt="Describe the attachment-backed visual region for retrieval augmentation.",
                        node_id=node_id,
                    )
                )
                attachments_by_node.setdefault(node_id, []).append(
                    {
                        "region_id": attachment.region_id,
                        "summary": attachment.insight.summary,
                        "labels": list(attachment.insight.labels),
                        "audit_path": attachment.audit_path,
                        "provider_identity": attachment.provider_identity,
                    }
                )
            except GatewayError as exc:
                attachments_by_node.setdefault(node_id, []).append(
                    {
                        "region_id": region_item["region"].region_id,
                        "summary": f"Visual enrichment failed: {exc.failure.category.value}",
                        "labels": ["gateway-failure"],
                        "audit_path": exc.audit_path,
                        "provider_identity": "noop",
                    }
                )
    if not attachments_by_node:
        trace.append("No visual attachments available for the selected nodes.")
    else:
        trace.append("Visual context populated from VisualEnrichmentAttachment outputs only.")
    return {
        "visual_attachments_by_node": attachments_by_node,
        "retrieval_trace": trace,
    }


def assemble_context_node(state: RAGState) -> dict[str, Any]:
    chunks = []
    citations = []
    for node_id in state["selected_leaf_ids"]:
        detail = node_details[node_id]
        attachments = state.get("visual_attachments_by_node", {}).get(node_id, [])
        chunks.append(
            {
                "node_id": node_id,
                "title": detail["title"],
                "path": " > ".join(detail["path"]),
                "page_span": detail["page_span"],
                "summary": detail["summary"],
                "keywords": detail["keywords"],
                "visual_attachments": attachments,
            }
        )
        citations.append(
            {
                "title": detail["title"],
                "pages": detail["page_span"],
            }
        )
    return {"context_chunks": chunks, "citations": citations}


def maybe_live_answer(system_prompt: str, user_prompt: str) -> dict[str, Any]:
    if TEXT_GATEWAY is None:
        return {
            "status": "skipped",
            "reason": "OPENROUTER_API_KEY not configured",
            "answer": None,
            "audit_path": None,
        }
    try:
        success = TEXT_GATEWAY.invoke(
            GatewayRequest(
                operation_name="cookbook_langgraph_answer",
                messages=(
                    LLMMessage(role=LLMRole.SYSTEM, content=system_prompt),
                    LLMMessage(role=LLMRole.USER, content=user_prompt),
                ),
                response_model=NotebookAnswerResponse,
                temperature=0.0,
                metadata={"notebook": "langgraph_rag_cookbook"},
            )
        )
    except GatewayError as exc:
        return {
            "status": "failure",
            "reason": exc.failure.category.value,
            "message": exc.failure.message,
            "answer": None,
            "audit_path": exc.audit_path,
            "provider_name": exc.failure.provider_name,
        }
    return {
        "status": "success",
        "answer": success.output.answer,
        "audit_path": success.audit_path,
        "provider_name": success.provider_name,
    }


def generate_answer_node(state: RAGState) -> dict[str, Any]:
    chunks = state["context_chunks"]
    query = state["query"]
    context_text = []
    for chunk in chunks:
        visuals = chunk.get("visual_attachments", [])
        visual_text = ""
        if visuals:
            visual_text = "\nVisual Insights: " + "; ".join(item["summary"] for item in visuals)
        context_text.append(
            f"[{chunk['title']}] pages {chunk['page_span']['start_page']}-{chunk['page_span']['end_page']}: {chunk['summary']}{visual_text}"
        )
    prompt = "\n".join(context_text)
    live_text_result = maybe_live_answer(
        "Answer using only the provided grounded context and mention page spans.",
        f"Query: {query}\n\nContext:\n{prompt}",
    )
    trace = state["retrieval_trace"][:]
    if live_text_result["status"] == "success" and live_text_result.get("answer"):
        trace.append("Generated final answer through GatewayService live text path.")
        return {
            "answer": live_text_result["answer"],
            "answer_mode": "gateway_live",
            "live_text_result": live_text_result,
            "retrieval_trace": trace,
        }
    if live_text_result["status"] == "failure":
        trace.append(
            f"Live text via GatewayService failed: {live_text_result['reason']}; using deterministic fallback."
        )
    else:
        trace.append(
            "Live text skipped because OPENROUTER_API_KEY is not configured; using deterministic fallback."
        )
    return {
        "answer": format_grounded_fallback_answer(query, chunks),
        "answer_mode": "deterministic_fallback",
        "live_text_result": live_text_result,
        "retrieval_trace": trace,
    }


rag_graph = None
query_results: list[dict[str, Any]] = []
if cookbook_failure is None:
    workflow = StateGraph(RAGState)
    workflow.add_node("load_artifacts", load_artifacts_node)
    workflow.add_node("select_sections", select_sections_node)
    workflow.add_node("drill_down", drill_down_node)
    workflow.add_node("enrich_visuals", enrich_visuals_node)
    workflow.add_node("assemble_context", assemble_context_node)
    workflow.add_node("generate_answer", generate_answer_node)
    workflow.set_entry_point("load_artifacts")
    workflow.add_edge("load_artifacts", "select_sections")
    workflow.add_edge("select_sections", "drill_down")
    workflow.add_edge("drill_down", "enrich_visuals")
    workflow.add_edge("enrich_visuals", "assemble_context")
    workflow.add_edge("assemble_context", "generate_answer")
    workflow.add_edge("generate_answer", END)
    rag_graph = workflow.compile()

    queries = [
        "What topics are covered in the first pages of this document?",
        "Which sections appear most relevant to visual or diagram-heavy content?",
    ]
    for query in queries:
        result = rag_graph.invoke({"query": query})
        query_results.append(
            {
                "query": query,
                "selected_l1_ids": result.get("selected_l1_ids", []),
                "selected_leaf_ids": result.get("selected_leaf_ids", []),
                "context_chunk_count": len(result.get("context_chunks", [])),
                "visual_attachment_count": sum(
                    len(items) for items in result.get("visual_attachments_by_node", {}).values()
                ),
                "answer_mode": result.get("answer_mode"),
                "live_text_status": result.get("live_text_result", {}).get("status"),
                "live_text_audit_path": result.get("live_text_result", {}).get("audit_path"),
                "answer_preview": result.get("answer", "")[:240],
                "trace": result.get("retrieval_trace", []),
            }
        )
else:
    query_results = [
        {
            "status": "blocked_tree_quality",
            "reason_code": cookbook_failure.reason_code,
            "tree_manifest_path": cookbook_failure.tree_manifest_path,
            "sample_rejected_titles": list(cookbook_failure.sample_rejected_titles),
            "message": cookbook_failure.message,
        }
    ]

print(json.dumps(query_results, indent=2))

In [ ]:
# execution
if rag_graph is None:
    result = {
        "status": "blocked_tree_quality",
        "failure": cookbook_failure.model_dump(mode="json")
        if cookbook_failure is not None
        else None,
    }
else:
    result = rag_graph.invoke({"query": "what is on first page?"})
print(json.dumps(result, indent=2, ensure_ascii=True))

In [ ]:
# inspect results
multimodal_audit_files = sorted((COOKBOOK_ARTIFACT_ROOT / "multimodal-audit").glob("*.json"))
llm_audit_files = (
    sorted((COOKBOOK_ARTIFACT_ROOT / "llm-audit").glob("*.json"))
    if (COOKBOOK_ARTIFACT_ROOT / "llm-audit").exists()
    else []
)
total_asset_backed_regions = sum(len(regions) for regions in node_visual_regions.values())
total_attachment_count = sum(result.get("visual_attachment_count", 0) for result in query_results)

graph_status = "blocked_tree_quality" if cookbook_failure is not None else "ready"
summary = {
    "pdf": {
        "path": str(PDF_PATH),
        "source_mode": PDF_SOURCE_MODE,
        "document_id": fingerprint.document_id,
    },
    "artifacts": {
        "acquisition_run_id": acquisition_manifest.acquisition_run_id,
        "tree_run_id": manifest_model.tree_run_id,
        "acquisition_manifest_path": str(
            Path(acquisition_manifest.artifact_root) / "manifest.json"
        ),
        "tree_manifest_path": tree_manifest_path,
    },
    "visual": {
        "asset_backed_region_count": total_asset_backed_regions,
        "nodes_with_visual_regions": sum(1 for regions in node_visual_regions.values() if regions),
        "attachment_count": total_attachment_count,
        "context_source": "attachments_only",
        "simulated_visual_descriptions": False,
        "multimodal_audit_count": len(multimodal_audit_files),
    },
    "graph": {
        "status": graph_status,
        "node_count": len(node_cards),
        "l1_section_count": len(l1_sections),
        "text_llm_enabled": TEXT_LLM_ENABLED,
        "quality_gate": quality_report.model_dump(mode="json"),
        "failure": cookbook_failure.model_dump(mode="json")
        if cookbook_failure is not None
        else None,
        "query_results": query_results,
        "llm_audit_count": len(llm_audit_files),
    },
}
print(json.dumps(summary, indent=2))

### Known Limitations

- Visual context in this notebook comes only from persisted attachment assets plus `VisualEnrichmentAttachment` outputs from the noop multimodal gateway.
- The notebook does not use a live vision provider; that remains outside the framework core.
- If `OPENROUTER_API_KEY` is absent, answer generation falls back to a deterministic text assembly so the notebook stays executable.
- If a local real-PDF tree looks polluted, the quality gate blocks answer generation and reports the failure instead of fabricating grounded output.
